In [1]:
import pandas as pd
import numpy as np

optimized_plan = pd.read_csv(
    "../data/predictions/optimized_block_plan.csv"
)

print(
    "Optimized plan shape:",
    optimized_plan.shape
)

display(optimized_plan)

Optimized plan shape: (3, 9)


,task_id,section_id,department,start_slot,end_slot,duration_minutes,required_manpower,maintenance_decision_score,predicted_delay_minutes
0,TMS001,NDL-MTJ-01,ENGINEERING,2,5,90,8,0.322000,0.0
1,SMMS001,MTJ-AGC-01,S&T,6,8,60,3,0.274500,0.0
2,TDMS001,GWL-JHS-01,TRACTION,8,10,60,5,0.270667,0.0


In [2]:
block_windows = [
    {
        "block_id": "B001",
        "start_slot": 1,
        "end_slot": 5
    },
    {
        "block_id": "B002",
        "start_slot": 6,
        "end_slot": 10
    }
]

SLOT_MINUTES = 30

print("Block windows loaded.")

Block windows loaded.


In [3]:
def find_block(start_slot, end_slot):

    for block in block_windows:

        if (
            start_slot >= block["start_slot"]
            and end_slot <= block["end_slot"]
        ):
            return block["block_id"]

    return "UNASSIGNED"


optimized_plan["block_id"] = optimized_plan.apply(
    lambda row: find_block(
        row["start_slot"],
        row["end_slot"]
    ),
    axis=1
)

display(
    optimized_plan[
        [
            "task_id",
            "section_id",
            "start_slot",
            "end_slot",
            "block_id"
        ]
    ]
)

,task_id,section_id,start_slot,end_slot,block_id
0,TMS001,NDL-MTJ-01,2,5,B001
1,SMMS001,MTJ-AGC-01,6,8,B002
2,TDMS001,GWL-JHS-01,8,10,B002


In [4]:
PLANNING_START_MINUTES = 0

def slot_to_time(slot):

    total_minutes = (
        PLANNING_START_MINUTES
        + slot * SLOT_MINUTES
    )

    hours = total_minutes // 60
    minutes = total_minutes % 60

    return f"{hours:02d}:{minutes:02d}"


optimized_plan["start_time"] = (
    optimized_plan["start_slot"]
    .apply(slot_to_time)
)

optimized_plan["end_time"] = (
    optimized_plan["end_slot"]
    .apply(slot_to_time)
)

display(
    optimized_plan[
        [
            "task_id",
            "section_id",
            "block_id",
            "start_time",
            "end_time",
            "duration_minutes"
        ]
    ]
)

,task_id,section_id,block_id,start_time,end_time,duration_minutes
0,TMS001,NDL-MTJ-01,B001,01:00,02:30,90
1,SMMS001,MTJ-AGC-01,B002,03:00,04:00,60
2,TDMS001,GWL-JHS-01,B002,04:00,05:00,60


In [5]:
controller_plan = optimized_plan[
    [
        "task_id",
        "section_id",
        "department",
        "block_id",
        "start_time",
        "end_time",
        "duration_minutes",
        "required_manpower",
        "maintenance_decision_score",
        "predicted_delay_minutes"
    ]
].copy()

controller_plan = controller_plan.sort_values(
    [
        "block_id",
        "start_time",
        "section_id"
    ]
).reset_index(drop=True)

controller_plan.insert(
    0,
    "plan_sequence",
    np.arange(
        1,
        len(controller_plan) + 1
    )
)

display(controller_plan)

,plan_sequence,task_id,section_id,department,block_id,start_time,end_time,duration_minutes,required_manpower,maintenance_decision_score,predicted_delay_minutes
0,1,TMS001,NDL-MTJ-01,ENGINEERING,B001,01:00,02:30,90,8,0.322000,0.0
1,2,SMMS001,MTJ-AGC-01,S&T,B002,03:00,04:00,60,3,0.274500,0.0
2,3,TDMS001,GWL-JHS-01,TRACTION,B002,04:00,05:00,60,5,0.270667,0.0


In [6]:
controller_plan.to_csv(
    "../data/predictions/controller_block_plan.csv",
    index=False
)

print(
    "Controller block plan saved successfully:"
)

print(
    "../data/predictions/controller_block_plan.csv"
)

Controller block plan saved successfully:
../data/predictions/controller_block_plan.csv


In [8]:
block_utilization = []

for block in block_windows:

    block_id = block["block_id"]

    available_slots = (
        block["end_slot"]
        - block["start_slot"]
    )

    available_minutes = (
        available_slots
        * SLOT_MINUTES
    )

    block_tasks = controller_plan[
        controller_plan["block_id"]
        == block_id
    ]

    used_minutes = (
        block_tasks["duration_minutes"]
        .sum()
    )

    utilization = (
        used_minutes
        / available_minutes
        if available_minutes > 0
        else 0
    )

    block_utilization.append({
        "block_id": block_id,
        "available_minutes": available_minutes,
        "used_minutes": used_minutes,
        "unused_minutes": (
            available_minutes
            - used_minutes
        ),
        "utilization": utilization
    })

block_utilization = pd.DataFrame(
    block_utilization
)

display(block_utilization)

,block_id,available_minutes,used_minutes,unused_minutes,utilization
0,B001,120,90,30,0.75
1,B002,120,120,0,1.00


In [9]:
total_available = (
    block_utilization["available_minutes"]
    .sum()
)

total_used = (
    block_utilization["used_minutes"]
    .sum()
)

overall_utilization = (
    total_used / total_available
    if total_available > 0
    else 0
)

print(
    "Overall block utilization:",
    round(
        overall_utilization * 100,
        2
    ),
    "%"
)

Overall block utilization: 87.5 %


In [10]:
department_coordination = (
    controller_plan
    .groupby("block_id")
    .agg(
        task_count=("task_id", "count"),
        department_count=(
            "department",
            "nunique"
        ),
        departments=(
            "department",
            lambda x: ", ".join(
                sorted(
                    x.dropna()
                    .astype(str)
                    .unique()
                )
            )
        ),
        sections=(
            "section_id",
            lambda x: ", ".join(
                sorted(
                    x.dropna()
                    .astype(str)
                    .unique()
                )
            )
        )
    )
    .reset_index()
)

display(department_coordination)

,block_id,task_count,department_count,departments,sections
0,B001,1,1,ENGINEERING,NDL-MTJ-01
1,B002,2,2,"S&T, TRACTION","GWL-JHS-01, MTJ-AGC-01"
